# NB-01 — Data Acquisition
## The Peacekeepers' Arms Race: Testing the Stability-Instability Paradox

This notebook downloads, inspects, and checkpoints every raw dataset used in the project.
Raw files are kept untouched in `data/raw/`; clean snapshots are written to `data/checkpoints/` as Parquet.

| # | Dataset | Provider | Granularity |
|---|---------|----------|-------------|
| 1 | UCDP Armed Conflict Dataset v25.1 | Uppsala / PRIO | conflict-year |
| 1b | UCDP Georeferenced Event Dataset (GED) v25.1 | Uppsala | event |
| 1c | UCDP Battle-Related Deaths v25.1 | Uppsala | dyad-year |
| 1d | UCDP Dyadic Dataset v25.1 | Uppsala | dyad-year |
| 2 | SIPRI Military Expenditure Database | SIPRI | country-year |
| 3 | SIPRI Arms Transfers (TIV) — by country | SIPRI | country-year |
| 3b | SIPRI Arms Transfers (TIV) — by category | SIPRI | category-year |
| 4 | World Development Indicators | World Bank API | country-year |
| 5 | V-Dem Core Dataset v16 | V-Dem Institute | country-year |
| 6 | National Material Capabilities (CINC) v6.0 | Correlates of War | state-year |

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import requests
import json
import hashlib
from pathlib import Path
from datetime import datetime
from src.config import UCDP_DIR, SIPRI_DIR, VDEM_DIR, WORLDBANK_DIR, COW_DIR, CHECKPOINT_DIR, RAW_DIR
from src.io_utils import save_checkpoint, load_checkpoint, checkpoint_exists

In [2]:
def write_provenance(filepath, source_url, notes=""):
    p = Path(filepath)
    sha256 = hashlib.sha256(p.read_bytes()).hexdigest()
    provenance = {
        "file": p.name,
        "source_url": source_url,
        "sha256": sha256,
        "downloaded_date": datetime.now().isoformat(),
        "notes": notes,
    }
    prov_path = p.parent / (p.stem + "_provenance.json")
    prov_path.write_text(json.dumps(provenance, indent=2))
    print(f"[provenance] written -> {prov_path.name}")

---
## 1. UCDP Armed Conflict Dataset (ACD)

**Source:** https://ucdp.uu.se/downloads/index.html#armedconflict
**Granularity:** conflict-year
**Key variables:** `conflict_id`, `year`, `type_of_conflict` (1–4), `intensity_level` (1 = minor, 2 = war),
`region`, `side_a`, `side_b`

Primary conflict-outcome variable for the Stability-Instability Paradox test.

In [3]:
path = UCDP_DIR / "ucdp_acd.csv"

if not path.exists():
    print("Missing:", path)
    print("Download from https://ucdp.uu.se/downloads/index.html#armedconflict")
    print("Place file at: data/raw/ucdp/ucdp_acd.csv")
else:
    acd = pd.read_csv(path)
    print("Shape:", acd.shape)
    print("Columns:", acd.columns.tolist())
    print(f"\nYear range: {acd['year'].min()} - {acd['year'].max()}")
    print("\ntype_of_conflict (1=extrasystemic, 2=interstate, 3=intrastate, 4=internationalised):")
    print(acd["type_of_conflict"].value_counts().sort_index())
    print("\nintensity_level (1=minor, 2=war):")
    print(acd["intensity_level"].value_counts().sort_index())

    if not checkpoint_exists(CHECKPOINT_DIR / "ucdp_acd_raw.parquet"):
        save_checkpoint(acd, CHECKPOINT_DIR / "ucdp_acd_raw.parquet")
    else:
        print("[skip] ucdp_acd_raw.parquet already exists")

    write_provenance(
        path,
        source_url="https://ucdp.uu.se/downloads/index.html#armedconflict",
        notes="UCDP/PRIO Armed Conflict Dataset v25.1, conflict-year level, 1946-2024",
    )

Shape: (2752, 28)
Columns: ['conflict_id', 'location', 'side_a', 'side_a_id', 'side_a_2nd', 'side_b', 'side_b_id', 'side_b_2nd', 'incompatibility', 'territory_name', 'year', 'intensity_level', 'cumulative_intensity', 'type_of_conflict', 'start_date', 'start_prec', 'start_date2', 'start_prec2', 'ep_end', 'ep_end_date', 'ep_end_prec', 'gwno_a', 'gwno_a_2nd', 'gwno_b', 'gwno_b_2nd', 'gwno_loc', 'region', 'version']

Year range: 1946 - 2024

type_of_conflict (1=extrasystemic, 2=interstate, 3=intrastate, 4=internationalised):
type_of_conflict
1     117
2     147
3    2004
4     484
Name: count, dtype: int64

intensity_level (1=minor, 2=war):
intensity_level
1    2066
2     686
Name: count, dtype: int64
[skip] ucdp_acd_raw.parquet already exists
[provenance] written -> ucdp_acd_provenance.json


---
## 1b. UCDP Georeferenced Event Dataset (GED)

**Source:** https://ucdp.uu.se/downloads/index.html#ged_global
**Granularity:** individual event (georeferenced)
**Key variables:** `id`, `year`, `date_start`, `type_of_violence`, `best` (best death estimate),
`latitude`, `longitude`, `country`

Enables subnational and event-density analysis of conflict severity.

In [4]:
path = UCDP_DIR / "ucdp_ged.csv"

if not path.exists():
    print("Missing:", path)
    print("Download UCDP GED Global v25.1 from:")
    print("  https://ucdp.uu.se/downloads/index.html#ged_global")
    print("Place file at: data/raw/ucdp/ucdp_ged.csv")
else:
    ged = pd.read_csv(path, low_memory=False)
    print("Shape:", ged.shape)
    print("Columns:", ged.columns.tolist())

    date_col = "date_start" if "date_start" in ged.columns else "year"
    print(f"\nDate range ({date_col}): {ged[date_col].min()} - {ged[date_col].max()}")

    if not checkpoint_exists(CHECKPOINT_DIR / "ucdp_ged_raw.parquet"):
        save_checkpoint(ged, CHECKPOINT_DIR / "ucdp_ged_raw.parquet")
    else:
        print("[skip] ucdp_ged_raw.parquet already exists")

    write_provenance(
        path,
        source_url="https://ucdp.uu.se/downloads/index.html#ged_global",
        notes="UCDP Georeferenced Event Dataset (GED) Global v25.1, event-level",
    )

Shape: (385918, 49)
Columns: ['id', 'relid', 'year', 'active_year', 'code_status', 'type_of_violence', 'conflict_dset_id', 'conflict_new_id', 'conflict_name', 'dyad_dset_id', 'dyad_new_id', 'dyad_name', 'side_a_dset_id', 'side_a_new_id', 'side_a', 'side_b_dset_id', 'side_b_new_id', 'side_b', 'number_of_sources', 'source_article', 'source_office', 'source_date', 'source_headline', 'source_original', 'where_prec', 'where_coordinates', 'where_description', 'adm_1', 'adm_2', 'latitude', 'longitude', 'geom_wkt', 'priogrid_gid', 'country', 'country_id', 'region', 'event_clarity', 'date_prec', 'date_start', 'date_end', 'deaths_a', 'deaths_b', 'deaths_civilians', 'deaths_unknown', 'best', 'high', 'low', 'gwnoa', 'gwnob']

Date range (date_start): 1989-01-01 00:00:00.000 - 2024-12-31 00:00:00.000
[skip] ucdp_ged_raw.parquet already exists
[provenance] written -> ucdp_ged_provenance.json


---
## 1c. UCDP Battle-Related Deaths (BRD)

**Source:** https://ucdp.uu.se/downloads/index.html#battlerelated
**Granularity:** dyad-year
**Key variables:** `conflict_id`, `dyad_id`, `year`, `bd_best`, `bd_low`, `bd_high`

Annual death estimates disaggregated by dyad; used as a conflict-severity measure.

In [5]:
path = UCDP_DIR / "ucdp_brd.csv"

if not path.exists():
    print("Missing:", path)
    print("Download UCDP Battle-Related Deaths Dataset v25.1 from:")
    print("  https://ucdp.uu.se/downloads/index.html#battlerelated")
    print("Place file at: data/raw/ucdp/ucdp_brd.csv")
else:
    brd = pd.read_csv(path)
    print("Shape:", brd.shape)
    print("Columns:", brd.columns.tolist())
    year_col = "year" if "year" in brd.columns else next(
        c for c in brd.columns if "year" in c.lower()
    )
    print(f"\nYear range: {brd[year_col].min()} - {brd[year_col].max()}")

    if not checkpoint_exists(CHECKPOINT_DIR / "ucdp_brd_raw.parquet"):
        save_checkpoint(brd, CHECKPOINT_DIR / "ucdp_brd_raw.parquet")
    else:
        print("[skip] ucdp_brd_raw.parquet already exists")

    write_provenance(
        path,
        source_url="https://ucdp.uu.se/downloads/index.html#battlerelated",
        notes="UCDP Battle-Related Deaths Dataset v25.1, dyad-year level",
    )

Shape: (1586, 25)
Columns: ['conflict_id', 'dyad_id', 'location_inc', 'side_a', 'side_a_id', 'side_a_2nd', 'side_b', 'side_b_id', 'side_b_2nd', 'incompatibility', 'territory_name', 'year', 'bd_best', 'bd_low', 'bd_high', 'type_of_conflict', 'battle_location', 'gwno_a', 'gwno_a_2nd', 'gwno_b', 'gwno_b_2nd', 'gwno_loc', 'gwno_battle', 'region', 'version']

Year range: 1989 - 2024
[skip] ucdp_brd_raw.parquet already exists
[provenance] written -> ucdp_brd_provenance.json


---
## 1d. UCDP Dyadic Dataset

**Source:** https://ucdp.uu.se/downloads/index.html#dyadic
**Granularity:** dyad-year
**Key variables:** `dyad_id`, `conflict_id`, `side_a`, `side_b`, `year`,
`incompatibility`, `intensity_level`, `type_of_conflict`

Disaggregates multi-party conflicts into bilateral dyads for dyadic-level modelling.

In [6]:
path = UCDP_DIR / "ucdp_dyadic.csv"

if not path.exists():
    print("Missing:", path)
    print("Download UCDP Dyadic Dataset v25.1 from:")
    print("  https://ucdp.uu.se/downloads/index.html#dyadic")
    print("Place file at: data/raw/ucdp/ucdp_dyadic.csv")
else:
    dyadic = pd.read_csv(path)
    print("Shape:", dyadic.shape)
    print("Columns:", dyadic.columns.tolist())
    print(f"\nYear range: {dyadic['year'].min()} - {dyadic['year'].max()}")

    if not checkpoint_exists(CHECKPOINT_DIR / "ucdp_dyadic_raw.parquet"):
        save_checkpoint(dyadic, CHECKPOINT_DIR / "ucdp_dyadic_raw.parquet")
    else:
        print("[skip] ucdp_dyadic_raw.parquet already exists")

    write_provenance(
        path,
        source_url="https://ucdp.uu.se/downloads/index.html#dyadic",
        notes="UCDP Dyadic Dataset v25.1, dyad-year level, 1946-2024",
    )

Shape: (3432, 25)
Columns: ['dyad_id', 'conflict_id', 'location', 'side_a', 'side_a_id', 'side_a_2nd', 'side_b', 'side_b_id', 'side_b_2nd', 'incompatibility', 'territory_name', 'year', 'intensity_level', 'type_of_conflict', 'start_date', 'start_prec', 'start_date2', 'start_prec2', 'gwno_a', 'gwno_a_2nd', 'gwno_b', 'gwno_b_2nd', 'gwno_loc', 'region', 'version']

Year range: 1946 - 2024
[skip] ucdp_dyadic_raw.parquet already exists
[provenance] written -> ucdp_dyadic_provenance.json


---
## 2. SIPRI Military Expenditure (MILEX)

**Source:** https://www.sipri.org/databases/milex
**Coverage:** 174 countries, 1949–2025, constant 2024 USD
**Key variable:** `milex_const2024_usd` — annual defence spending

SIPRI encodes suppressed/missing values as `...`, `xx`, and similar strings.
These are coerced to `NaN` via `pd.to_numeric(errors='coerce')` before checkpointing.
Zero-expenditure rows are retained but flagged below for verification.

In [7]:
path = SIPRI_DIR / "sipri_milex.xlsx"

if not path.exists():
    print("Missing:", path)
    print("Download from https://www.sipri.org/databases/milex")
    print("Place file at: data/raw/sipri/sipri_milex.xlsx")
else:
    milex_raw = pd.read_excel(
        path,
        sheet_name="Constant (2024) US$",
        skiprows=5,
    )

    # Drop metadata columns, keep Country + integer year columns
    milex = milex_raw.drop(
        columns=[c for c in ["Unnamed: 1", "Notes"] if c in milex_raw.columns]
    )
    year_cols = [c for c in milex.columns if isinstance(c, int)]
    milex = milex.dropna(subset=year_cols, how="all").dropna(subset=["Country"])
    milex = milex.reset_index(drop=True)

    # Melt to long format
    milex_long = milex.melt(
        id_vars=["Country"],
        value_vars=year_cols,
        var_name="year",
        value_name="milex_const2024_usd",
    )
    milex_long["milex_const2024_usd"] = pd.to_numeric(
        milex_long["milex_const2024_usd"], errors="coerce"
    )
    milex_long = milex_long.dropna(subset=["milex_const2024_usd"])
    milex_long["year"] = milex_long["year"].astype(int)
    milex_long = milex_long.sort_values(["Country", "year"]).reset_index(drop=True)

    print("Shape:", milex_long.shape)
    print("Dtypes:\n", milex_long.dtypes)
    print("\nDescribe:")
    print(milex_long["milex_const2024_usd"].describe())

    print("\nZero-expenditure rows (sample):")
    print(milex_long[milex_long["milex_const2024_usd"] == 0][["Country", "year"]].head(20))

    if not checkpoint_exists(CHECKPOINT_DIR / "sipri_milex_long_raw.parquet"):
        save_checkpoint(milex_long, CHECKPOINT_DIR / "sipri_milex_long_raw.parquet")
    else:
        print("[skip] sipri_milex_long_raw.parquet already exists")

    write_provenance(
        path,
        source_url="https://www.sipri.org/databases/milex",
        notes="SIPRI Military Expenditure Database, Constant (2024) US$, 174 countries, 1949-2025",
    )

Shape: (8435, 3)
Dtypes:
 Country                    str
year                     int64
milex_const2024_usd    float64
dtype: object

Describe:
count    8.435000e+03
mean     1.232535e+04
std      6.986929e+04
min      0.000000e+00
25%      1.297652e+02
50%      9.045244e+02
75%      4.708165e+03
max      1.061674e+06
Name: milex_const2024_usd, dtype: float64

Zero-expenditure rows (sample):
         Country  year
1695  Costa Rica  1949
1696  Costa Rica  1950
1697  Costa Rica  1951
1698  Costa Rica  1952
1699  Costa Rica  1953
1700  Costa Rica  1954
1701  Costa Rica  1955
1702  Costa Rica  1956
1703  Costa Rica  1957
1704  Costa Rica  1958
1705  Costa Rica  1959
1706  Costa Rica  1960
1707  Costa Rica  1961
1708  Costa Rica  1962
1709  Costa Rica  1963
1710  Costa Rica  1964
1711  Costa Rica  1965
1712  Costa Rica  1966
1713  Costa Rica  1967
1714  Costa Rica  1968
[skip] sipri_milex_long_raw.parquet already exists
[provenance] written -> sipri_milex_provenance.json


---
## 3. SIPRI Arms Transfers — Transfer Register (TIV)

**Source:** https://www.sipri.org/databases/armstransfers  
**File:** `sipri_tiv_register.csv` — full deal-level transfer register  
**Coverage:** 1950–2025  
**Key variables:** `supplier`, `recipient`, `weapon_category`, `year`, `tiv`

Each row is one delivery batch from a single supplier–recipient deal.
This replaces the pre-aggregated `sipri_tiv_by_country.csv` and enables richer disaggregation.
The aggregation cell below derives:

- **`sipri_tiv_by_country_year`** — recipient × year totals (`tiv_imports_total`)
- **`sipri_tiv_by_country_year_category`** — recipient × year × weapon category pivot

In [8]:
path = SIPRI_DIR / "sipri_tiv_register.csv"

if not path.exists():
    print("Missing:", path)
    print("Download the full transfer register from:")
    print("  https://www.sipri.org/databases/armstransfers")
    print("Place file at: data/raw/sipri/sipri_tiv_register.csv")
else:
    # Peek at first 15 lines to confirm header row
    print("=== Header peek (first 15 lines) ===")
    with open(path, encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            print(f"  {i:2}: {line[:120].rstrip()}")
            if i >= 14:
                break

    reg = pd.read_csv(path, skiprows=11, header=0, low_memory=False)
    print("\nShape:", reg.shape)
    print("Columns:", reg.columns.tolist())

    keep = {
        "Supplier":            "supplier",
        "Recipient":           "recipient",
        "Armament category":   "weapon_category",
        "Delivery year":       "year",
        "TIV delivery values": "tiv",
    }
    reg = reg[list(keep.keys())].rename(columns=keep)

    reg["tiv"]  = pd.to_numeric(reg["tiv"],  errors="coerce")
    reg["year"] = pd.to_numeric(reg["year"], errors="coerce")
    reg = reg.dropna(subset=["tiv", "year"])
    reg["year"] = reg["year"].astype(int)
    reg = reg.sort_values(["recipient", "year"]).reset_index(drop=True)

    print("\nCleaned shape:", reg.shape)
    print("Unique weapon categories:", sorted(reg["weapon_category"].dropna().unique().tolist()))
    print(f"Year range: {reg['year'].min()} - {reg['year'].max()}")

    if not checkpoint_exists(CHECKPOINT_DIR / "sipri_tiv_register_raw.parquet"):
        save_checkpoint(reg, CHECKPOINT_DIR / "sipri_tiv_register_raw.parquet")
    else:
        print("[skip] sipri_tiv_register_raw.parquet already exists")

    write_provenance(
        path,
        source_url="https://www.sipri.org/databases/armstransfers",
        notes="SIPRI Arms Transfers Database, full deal-level register, 1950-2025",
    )

=== Header peek (first 15 lines) ===
   0: Transfers of major conventional arms from All countries  to All countries . Deals with deliveries made for the year rang
   1: A '?' in a column indicates uncertain data. The 'Number delivered' and the 'Year(s) of deliveries' refer only to deliver
   2: An empty field for 'Number ordered' indicates that data is not yet available.
   3: SIPRI trend-indicator values (TIVs) are in millions.
   4: An empty field for 'SIPRI TIV for total order' indicates that data (on the number ordered and/or the TIV per unit) is no
   5: A '0' for 'SIPRI TIV of delivered weapons' indicates that the volume of deliveries is between 0 and 0.5 million SIPRI TI
   6: Figures may not add up to stated totals due to the conventions of rounding.
   7: For the method used for the SIPRI TIV and explanations of the conventions; abbreviations and acronyms see <https://www.s
   8: 
   9: Source: SIPRI Arms Transfers Database (c) SIPRI.
  10: Data generated: 13 May 2026 4:50:46

In [9]:
if not checkpoint_exists(CHECKPOINT_DIR / "sipri_tiv_register_raw.parquet"):
    print("[skip] sipri_tiv_register_raw.parquet not found — run Section 3 first")
else:
    reg = load_checkpoint(CHECKPOINT_DIR / "sipri_tiv_register_raw.parquet")

    # Aggregation 1: recipient x year -> total TIV imports
    by_country_year = (
        reg.groupby(["recipient", "year"], as_index=False)["tiv"]
        .sum()
        .rename(columns={"tiv": "tiv_imports_total"})
        .sort_values(["recipient", "year"])
        .reset_index(drop=True)
    )
    print("by_country_year shape:", by_country_year.shape)
    print(by_country_year.head(3))

    # Aggregation 2: recipient x year x weapon_category -> pivot
    by_cwc = reg.groupby(["recipient", "year", "weapon_category"], as_index=False)["tiv"].sum()
    by_country_year_cat = by_cwc.pivot_table(
        index=["recipient", "year"],
        columns="weapon_category",
        values="tiv",
        aggfunc="sum",
        fill_value=0,
    ).reset_index()
    by_country_year_cat.columns.name = None

    # Normalise column names: lowercase, spaces/hyphens -> underscores
    by_country_year_cat.columns = [
        c if c in ("recipient", "year")
        else c.lower().replace(" ", "_").replace("-", "_")
        for c in by_country_year_cat.columns
    ]
    print("\nby_country_year_cat shape:", by_country_year_cat.shape)
    print("Category columns:", [c for c in by_country_year_cat.columns if c not in ("recipient", "year")])

    if not checkpoint_exists(CHECKPOINT_DIR / "sipri_tiv_by_country_year.parquet"):
        save_checkpoint(by_country_year, CHECKPOINT_DIR / "sipri_tiv_by_country_year.parquet")
    else:
        print("[skip] sipri_tiv_by_country_year.parquet already exists")

    if not checkpoint_exists(CHECKPOINT_DIR / "sipri_tiv_by_country_year_category.parquet"):
        save_checkpoint(by_country_year_cat, CHECKPOINT_DIR / "sipri_tiv_by_country_year_category.parquet")
    else:
        print("[skip] sipri_tiv_by_country_year_category.parquet already exists")

[checkpoint] loaded ← sipri_tiv_register_raw.parquet  (60,789 rows)
by_country_year shape: (8213, 3)
             recipient  year  tiv_imports_total
0  ANC (South Africa)*  1988               0.24
1          Afghanistan  1955               4.00
2          Afghanistan  1956               3.60

by_country_year_cat shape: (8213, 13)
Category columns: ['air_defence_systems', 'aircraft', 'armoured_vehicles', 'artillery', 'engines', 'missiles', 'naval_weapons', 'other', 'satellites', 'sensors', 'ships']
[skip] sipri_tiv_by_country_year.parquet already exists
[skip] sipri_tiv_by_country_year_category.parquet already exists


---
## 3b. SIPRI Arms Transfers (TIV) — by Weapon Category

**Source:** https://www.sipri.org/databases/armstransfers
**Coverage:** 1950–2025
**Categories (after normalisation):**
`aircraft`, `armoured_vehicles`, `artillery`, `engines`, `missiles`,
`naval_weapons`, `sensors`, `air_defence_systems`, `satellites`, `ships`, `other`

Rows in the raw file are weapon categories; columns are years.
Transposed here to a year × category wide format.

In [10]:
path = SIPRI_DIR / "sipri_tiv_by_category.csv"

if not path.exists():
    print("Missing:", path)
    print("Download from https://www.sipri.org/databases/armstransfers")
    print("Place file at: data/raw/sipri/sipri_tiv_by_category.csv")
else:
    tiv_cat = pd.read_csv(path, skiprows=9, header=0)
    tiv_cat = tiv_cat.rename(columns={tiv_cat.columns[0]: "category"})
    tiv_cat = tiv_cat[tiv_cat["category"] != "Total"].copy()

    drop_cols = [
        c for c in tiv_cat.columns
        if any(kw in str(c).lower() for kw in ["total", "percentage", "sum"])
    ]
    tiv_cat = tiv_cat.drop(columns=drop_cols)

    year_cols = [c for c in tiv_cat.columns if c != "category" and str(c).isdigit()]

    tiv_wide = tiv_cat.set_index("category")[year_cols].T
    tiv_wide.index.name = "year"
    tiv_wide = tiv_wide.reset_index()

    col_map = {
        "Air-defence systems": "air_defence_systems",
        "Aircraft":            "aircraft",
        "Armoured vehicles":   "armoured_vehicles",
        "Artillery":           "artillery",
        "Engines":             "engines",
        "Missiles":            "missiles",
        "Naval weapons":       "naval_weapons",
        "Other":               "other",
        "Satellites":          "satellites",
        "Sensors":             "sensors",
        "Ships":               "ships",
    }
    tiv_wide = tiv_wide.rename(columns=col_map)

    for col in [c for c in tiv_wide.columns if c != "year"]:
        tiv_wide[col] = pd.to_numeric(tiv_wide[col], errors="coerce")
    tiv_wide["year"] = tiv_wide["year"].astype(int)
    tiv_wide = tiv_wide.sort_values("year").reset_index(drop=True)

    print("Shape:", tiv_wide.shape)
    print("Columns:", tiv_wide.columns.tolist())
    print(tiv_wide.head(3))

    if not checkpoint_exists(CHECKPOINT_DIR / "sipri_tiv_category_raw.parquet"):
        save_checkpoint(tiv_wide, CHECKPOINT_DIR / "sipri_tiv_category_raw.parquet")
    else:
        print("[skip] sipri_tiv_category_raw.parquet already exists")

    write_provenance(
        path,
        source_url="https://www.sipri.org/databases/armstransfers",
        notes="SIPRI Arms Transfers Database, TIV by weapon category, 1950-2025",
    )

Shape: (76, 12)
Columns: ['year', 'air_defence_systems', 'aircraft', 'armoured_vehicles', 'artillery', 'engines', 'missiles', 'naval_weapons', 'other', 'satellites', 'sensors', 'ships']
category  year  air_defence_systems  aircraft  armoured_vehicles  artillery  \
0         1950                  NaN    5263.0             1067.0      409.0   
1         1951                  NaN    6772.0             1008.0      527.0   
2         1952                 42.0   11735.0             2525.0      633.0   

category  engines  missiles  naval_weapons  other  satellites  sensors   ships  
0           442.0       NaN            1.0    NaN         NaN     55.0   914.0  
1           684.0       NaN           12.0    NaN         NaN     15.0  2460.0  
2           729.0       NaN            NaN    NaN         NaN     98.0   859.0  
[skip] sipri_tiv_category_raw.parquet already exists
[provenance] written -> sipri_tiv_by_category_provenance.json


---
## 4. World Bank World Development Indicators (WDI)

**Source:** World Bank API via `wbdata`
**Coverage:** All countries / aggregates, 1960–2024
**Key variables:**

| Indicator | Column | Description |
|-----------|--------|-------------|
| `NY.GDP.MKTP.CD` | `gdp_usd` | GDP, current USD |
| `NY.GDP.PCAP.CD` | `gdp_per_capita` | GDP per capita, current USD |
| `SP.POP.TOTL` | `population` | Total population |

> Aggregate rows (World, OECD, Euro area, High income, etc.) are included here and will be
> filtered out in NB-02 during the panel-construction step.

In [11]:
import wbdata

if not checkpoint_exists(CHECKPOINT_DIR / "worldbank_wdi_raw.parquet"):
    indicators = {
        "NY.GDP.MKTP.CD": "gdp_usd",
        "NY.GDP.PCAP.CD": "gdp_per_capita",
        "SP.POP.TOTL":    "population",
    }
    wb_raw = wbdata.get_dataframe(indicators, date=("1960", "2024"))
    wb_long = wb_raw.reset_index().rename(columns={"date": "year"})
    wb_long["year"] = wb_long["year"].astype(int)
    wb_long = wb_long.sort_values(["country", "year"]).reset_index(drop=True)

    print("Shape:", wb_long.shape)
    print("Columns:", wb_long.columns.tolist())
    print(wb_long.head(3))
    print("\nNote: aggregate rows (World, OECD, Euro area, etc.) will be filtered in NB-02.")

    save_checkpoint(wb_long, CHECKPOINT_DIR / "worldbank_wdi_raw.parquet")
else:
    wb_long = load_checkpoint(CHECKPOINT_DIR / "worldbank_wdi_raw.parquet")
    print("Shape:", wb_long.shape)
    print("Note: aggregate rows (World, OECD, Euro area, etc.) will be filtered in NB-02.")

[checkpoint] loaded ← worldbank_wdi_raw.parquet  (17,195 rows)
Shape: (17195, 5)
Note: aggregate rows (World, OECD, Euro area, etc.) will be filtered in NB-02.


---
## 5. V-Dem Core Dataset

**Source:** https://v-dem.net/data/the-v-dem-dataset/
**File:** `V-Dem-CY-Core-v16.csv` (Country-Year Core)
**Coverage:** ~180 countries, 1789–2024
**Key variable:** `v2x_polyarchy` — Electoral Democracy Index (0–1 continuous scale)

Used as a regime-type control variable. Democratic states may exhibit different conflict
propensities under the Stability-Instability Paradox.

In [12]:
path = VDEM_DIR / "V-Dem-CY-Core-v16.csv"

if not path.exists():
    print("Missing:", path)
    print("Download Country-Year V-Dem Core CSV (v16) from:")
    print("  https://v-dem.net/data/the-v-dem-dataset/")
    print("Place file at: data/raw/vdem/V-Dem-CY-Core-v16.csv")
else:
    vdem_raw = pd.read_csv(path, low_memory=False)
    print("Raw shape:", vdem_raw.shape)

    keep_cols = ["country_name", "country_text_id", "year", "v2x_polyarchy"]
    vdem = vdem_raw[keep_cols].copy()
    vdem = vdem[(vdem["year"] >= 1946) & (vdem["year"] <= 2024)]
    vdem = vdem.sort_values(["country_name", "year"]).reset_index(drop=True)

    print("Filtered shape:", vdem.shape)
    print(vdem.head(3))

    if not checkpoint_exists(CHECKPOINT_DIR / "vdem_raw.parquet"):
        save_checkpoint(vdem, CHECKPOINT_DIR / "vdem_raw.parquet")
    else:
        print("[skip] vdem_raw.parquet already exists")

    write_provenance(
        path,
        source_url="https://v-dem.net/data/the-v-dem-dataset/",
        notes="V-Dem Country-Year Core v16, v2x_polyarchy Electoral Democracy Index, 1789-2024",
    )

Raw shape: (28092, 1908)
Filtered shape: (13080, 4)
  country_name country_text_id  year  v2x_polyarchy
0  Afghanistan             AFG  1946          0.089
1  Afghanistan             AFG  1947          0.089
2  Afghanistan             AFG  1948          0.089
[skip] vdem_raw.parquet already exists
[provenance] written -> V-Dem-CY-Core-v16_provenance.json


---
## 6. COW National Material Capabilities (CINC)

**Source:** https://correlatesofwar.org/data-sets/national-material-capabilities/
**File:** any CSV containing `NMC` in the name under `data/raw/cow/`
**Coverage:** ~200 states, 1816–2016 (NMC v6.0)
**Key variables:** `stateabb`, `year`, `milper` (military personnel), `milex`,
`cinc` (Composite Index of National Capability)

The CINC score is the primary operationalisation of *relative state power* in this study.
Note that the dataset ends at 2016; coverage for 2017–2024 will need imputation or an alternative source.

In [13]:
nmc_files = sorted(f for f in COW_DIR.glob("*.csv") if "NMC" in f.name or "nmc" in f.name.lower())

if not nmc_files:
    print("No NMC CSV found in", COW_DIR)
    print("Download NMC v6.0 from:")
    print("  https://correlatesofwar.org/data-sets/national-material-capabilities/")
    print("Place CSV file in: data/raw/cow/")
else:
    path = nmc_files[0]
    print(f"Loading: {path.name}")

    cow_raw = pd.read_csv(path)
    print("Raw shape:", cow_raw.shape)

    keep_cols = ["stateabb", "year", "milper", "milex", "cinc"]
    cow = cow_raw[[c for c in keep_cols if c in cow_raw.columns]].copy()
    cow = cow.sort_values(["stateabb", "year"]).reset_index(drop=True)

    print("Shape:", cow.shape)
    print(f"Year range: {cow['year'].min()} - {cow['year'].max()}")
    print("Note: CINC data ends at 2016 in NMC v6.0.")
    print(cow.head(3))

    if not checkpoint_exists(CHECKPOINT_DIR / "cow_cinc_raw.parquet"):
        save_checkpoint(cow, CHECKPOINT_DIR / "cow_cinc_raw.parquet")
    else:
        print("[skip] cow_cinc_raw.parquet already exists")

    write_provenance(
        path,
        source_url="https://correlatesofwar.org/data-sets/national-material-capabilities/",
        notes="COW NMC v6.0, CINC scores 1816-2016",
    )

Loading: NMC-60-abridged.csv
Raw shape: (15951, 11)
Shape: (15951, 5)
Year range: 1816 - 2016
Note: CINC data ends at 2016 in NMC v6.0.
  stateabb  year  milper  milex      cinc
0      AAB  1981       0      0  0.000002
1      AAB  1982       0     -9  0.000005
2      AAB  1983       0     -9  0.000003
[skip] cow_cinc_raw.parquet already exists
[provenance] written -> NMC-60-abridged_provenance.json


---
## 7. Checkpoint Summary

Row-count and year-range audit across all saved checkpoints.
Datasets marked *not downloaded yet* require manual file placement before NB-02.

In [14]:
def _summary_row(label, cp_path, year_col="year"):
    if not checkpoint_exists(cp_path):
        return f"{label:<30}  not downloaded yet"
    df = pd.read_parquet(cp_path)
    n = len(df)
    yr = (
        f"({int(df[year_col].min())}-{int(df[year_col].max())})"
        if year_col in df.columns else ""
    )
    return f"{label:<30}  {n:>8,} rows  {yr}"

checkpoints = [
    ("UCDP ACD",                   CHECKPOINT_DIR / "ucdp_acd_raw.parquet"),
    ("UCDP GED",                   CHECKPOINT_DIR / "ucdp_ged_raw.parquet"),
    ("UCDP BRD",                   CHECKPOINT_DIR / "ucdp_brd_raw.parquet"),
    ("UCDP Dyadic",                CHECKPOINT_DIR / "ucdp_dyadic_raw.parquet"),
    ("SIPRI MILEX",                CHECKPOINT_DIR / "sipri_milex_long_raw.parquet"),
    ("SIPRI TIV register",         CHECKPOINT_DIR / "sipri_tiv_register_raw.parquet"),
    ("  -> by country-year",       CHECKPOINT_DIR / "sipri_tiv_by_country_year.parquet"),
    ("  -> by country-year-cat",   CHECKPOINT_DIR / "sipri_tiv_by_country_year_category.parquet"),
    ("SIPRI TIV category",         CHECKPOINT_DIR / "sipri_tiv_category_raw.parquet"),
    ("World Bank WDI",             CHECKPOINT_DIR / "worldbank_wdi_raw.parquet"),
    ("V-Dem",                      CHECKPOINT_DIR / "vdem_raw.parquet"),
    ("COW CINC",                   CHECKPOINT_DIR / "cow_cinc_raw.parquet"),
]

print("=== NB01 Checkpoint Summary ===")
print("Note: sipri_tiv_by_country.csv (pre-aggregated totals) is no longer used.")
print("      TIV data now sourced from the full deal-level transfer register.\n")
for label, path in checkpoints:
    print(_summary_row(label, path))

=== NB01 Checkpoint Summary ===
Note: sipri_tiv_by_country.csv (pre-aggregated totals) is no longer used.
      TIV data now sourced from the full deal-level transfer register.

UCDP ACD                           2,752 rows  (1946-2024)
UCDP GED                         385,918 rows  (1989-2024)
UCDP BRD                           1,586 rows  (1989-2024)
UCDP Dyadic                        3,432 rows  (1946-2024)
SIPRI MILEX                        8,435 rows  (1949-2025)
SIPRI TIV register                60,789 rows  (1950-2025)
  -> by country-year               8,213 rows  (1950-2025)
  -> by country-year-cat           8,213 rows  (1950-2025)
SIPRI TIV category                    76 rows  (1950-2025)
World Bank WDI                    17,195 rows  (1960-2024)
V-Dem                             13,080 rows  (1946-2024)
COW CINC                          15,951 rows  (1816-2016)
